# BioX-A Biostimulant — End-to-End AI/ML POC with Experiments Management

This notebook is a **self-contained proof-of-concept** demonstrating how BioX can use an AI/ML platform to:

1. Generate biologically plausible **synthetic agronomic data** (fields, treatments, satellite indices, weather, soil, formulations, outcomes) into **Postgres**
2. Perform **EDA** and **feature engineering** (difference-in-differences on vegetation indices, stress-window indicators, treatment-by-stress interactions)
3. Train three core models: **yield prediction**, **causal uplift**, and a **deterministic recommendation engine**
4. Wrap everything in an **experiments management layer** (MLflow + a Postgres `biox_experiments` schema) organized as **Project → Experiment → Run → Evaluation → Metrics**
5. Produce **field-level recommendations** with ROI, confidence, and explainable evidence

> **Important framing:** The synthetic data embeds a *known ground-truth* treatment effect (BioX-A helps under **moderate water stress**, is neutral under no stress, and does not help under severe drought/salinity or wrong crop stage). The POC proves the **platform can recover the correct response pattern** — it does **not** prove product efficacy. All outputs are framed as *"based on synthetic POC evidence and model behavior."*

---

## Run order

1. **Install & start Postgres** (see the README cell at the bottom, or `SETUP` below). Ensure it is reachable at the connection string in the **Config** cell.
2. **Run this notebook top-to-bottom** (`Kernel → Restart & Run All`). It is idempotent — safe to re-run.
3. **Launch the dashboard**: `streamlit run biox_experiments_dashboard.py`

The dashboard is a **read-only viewer** — it requires Postgres to be running and this notebook to have been run at least once.

## 0. Dependencies

Run this once to install the required packages. Safe to re-run (pip will skip already-installed packages).

In [ ]:
# Install dependencies (uncomment to run). EconML is optional; the notebook falls back
# to a statsmodels-based causal estimator if EconML is unavailable.
# %pip install pandas numpy sqlalchemy psycopg2-binary scikit-learn xgboost lightgbm \
#     shap statsmodels matplotlib seaborn plotly mlflow streamlit econml

## 1. Config & imports

Edit `PG_CONFIG` to match your local Postgres. MLflow uses a local `./mlruns` folder by default (no server needed).

In [ ]:
import os
import json
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG -- edit to match your environment
# ----------------------------------------------------------------------------
PG_CONFIG = {
    "host": os.environ.get("BIOX_PG_HOST", "localhost"),
    "port": os.environ.get("BIOX_PG_PORT", "5432"),
    "dbname": os.environ.get("BIOX_PG_DB", "biox_poc"),
    "user": os.environ.get("BIOX_PG_USER", "postgres"),
    "password": os.environ.get("BIOX_PG_PASSWORD", "postgres"),
}

# SQLAlchemy connection URL
PG_URL = (
    f"postgresql+psycopg2://{PG_CONFIG['user']}:{PG_CONFIG['password']}"
    f"@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}"
)

# MLflow local tracking store (no server required)
MLFLOW_TRACKING_URI = os.environ.get(
    "BIOX_MLFLOW_URI", f"file://{os.path.abspath('./mlruns')}"
)
# Alternative (Postgres-backed MLflow backend store):
# MLFLOW_TRACKING_URI = f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}"

ARTIFACT_DIR = os.path.abspath("./artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_FIELDS = 500
N_SITES = 8
CROP = "tomato"
REGION = "tropical_asia"
PRODUCT = "BioX-A"

# Postgres schema for the experiments management layer
EXP_SCHEMA = "biox_experiments"

print("Config loaded.")
print(f"  Postgres URL : postgresql+psycopg2://{PG_CONFIG['user']}:***@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}")
print(f"  MLflow URI   : {MLFLOW_TRACKING_URI}")
print(f"  Artifacts    : {ARTIFACT_DIR}")

In [ ]:
# Create the SQLAlchemy engine and verify connectivity.
# If this fails, ensure Postgres is running and the database in PG_CONFIG exists:
#   createdb biox_poc      (or)     CREATE DATABASE biox_poc;
engine = create_engine(PG_URL, pool_pre_ping=True)

try:
    with engine.connect() as conn:
        version = conn.execute(text("SELECT version();")).scalar()
    print("Connected to Postgres:")
    print(" ", version)
except Exception as e:
    print("ERROR: could not connect to Postgres.")
    print("  Check that Postgres is running and the database exists.")
    print("  Details:", e)
    raise

## 2. Agronomic schema (7 tables)

Creates the raw synthetic data model in the `public` schema. Idempotent: drops and recreates the POC tables on each run so the notebook is fully reproducible.

In [ ]:
AGRO_DDL = """
DROP TABLE IF EXISTS outcomes CASCADE;
DROP TABLE IF EXISTS satellite_indices CASCADE;
DROP TABLE IF EXISTS weather CASCADE;
DROP TABLE IF EXISTS soil CASCADE;
DROP TABLE IF EXISTS treatments CASCADE;
DROP TABLE IF EXISTS formulations CASCADE;
DROP TABLE IF EXISTS fields CASCADE;
DROP TABLE IF EXISTS feature_store CASCADE;

CREATE TABLE formulations (
    product_id        TEXT PRIMARY KEY,
    product_name      TEXT NOT NULL,
    active_ingredient TEXT NOT NULL,
    concentration_pct NUMERIC(5,2) NOT NULL,
    description       TEXT
);

CREATE TABLE fields (
    field_id             TEXT PRIMARY KEY,
    site_id              TEXT NOT NULL,
    crop                 TEXT NOT NULL,
    variety              TEXT,
    lat                  NUMERIC(8,5),
    lon                  NUMERIC(8,5),
    area_ha              NUMERIC(6,2),
    irrigation_type      TEXT,
    historical_yield_avg NUMERIC(6,2)
);

CREATE TABLE soil (
    field_id           TEXT PRIMARY KEY REFERENCES fields(field_id) ON DELETE CASCADE,
    ph                 NUMERIC(4,2),
    ec_ds_m            NUMERIC(5,2),
    organic_matter_pct NUMERIC(5,2),
    texture_class      TEXT,
    cec                NUMERIC(6,2)
);

CREATE TABLE treatments (
    field_id           TEXT PRIMARY KEY REFERENCES fields(field_id) ON DELETE CASCADE,
    treatment_flag     INTEGER NOT NULL,          -- 1 = BioX-A, 0 = control
    product_id         TEXT REFERENCES formulations(product_id),
    dose_ml_ha         NUMERIC(8,2),
    application_method TEXT,                       -- foliar / soil_drench / fertigation
    crop_stage         TEXT,                       -- vegetative / flowering / fruit_set / ripening
    application_date   DATE
);

CREATE TABLE weather (
    weather_id    BIGSERIAL PRIMARY KEY,
    field_id      TEXT NOT NULL REFERENCES fields(field_id) ON DELETE CASCADE,
    period        TEXT NOT NULL,                   -- pre / post
    rainfall_mm   NUMERIC(7,2),
    temp_avg_c    NUMERIC(5,2),
    temp_max_c    NUMERIC(5,2),
    vpd_kpa       NUMERIC(5,2),
    drought_index NUMERIC(5,3)                     -- 0 (wet) .. 1 (severe drought)
);

CREATE TABLE satellite_indices (
    sat_id             BIGSERIAL PRIMARY KEY,
    field_id           TEXT NOT NULL REFERENCES fields(field_id) ON DELETE CASCADE,
    period             TEXT NOT NULL,              -- pre / post
    ndvi               NUMERIC(5,3),
    ndre               NUMERIC(5,3),
    ndwi               NUMERIC(5,3),
    savi               NUMERIC(5,3),
    sentinel1_vv       NUMERIC(7,3),
    sentinel1_vh       NUMERIC(7,3),
    sentinel1_vv_vh    NUMERIC(7,3),
    sentinel3_lst      NUMERIC(6,2)                -- land surface temp (C)
);

CREATE TABLE outcomes (
    field_id             TEXT PRIMARY KEY REFERENCES fields(field_id) ON DELETE CASCADE,
    yield_t_ha           NUMERIC(6,3),
    marketable_yield_t_ha NUMERIC(6,3),
    quality_score        NUMERIC(5,2),
    product_cost_usd     NUMERIC(8,2),
    application_cost_usd NUMERIC(8,2),
    crop_price_usd_t     NUMERIC(8,2)
);

CREATE INDEX IF NOT EXISTS idx_sat_field ON satellite_indices(field_id);
CREATE INDEX IF NOT EXISTS idx_weather_field ON weather(field_id);
"""

with engine.begin() as conn:
    for stmt in [s for s in AGRO_DDL.split(";") if s.strip()]:
        conn.execute(text(stmt))

print("Agronomic schema created (7 tables + indexes).")

## 3. Synthetic data generator (known ground-truth effect)

Generates ~500 tomato fields across 8 sites in tropical Asia. Roughly half are treated with BioX-A, half are control.

**Embedded ground-truth treatment effect** (this is what the models must recover):

| Stress condition | BioX-A effect on yield |
|------------------|------------------------|
| Moderate water stress | **+0.45 t/ha** (strong benefit) |
| No / low stress | ~+0.05 t/ha (negligible) |
| Severe drought | ~0 (no benefit) |
| High salinity | ~-0.05 t/ha (slightly harmful) |
| Wrong crop stage (not flowering/fruit_set) | benefit multiplied by 0.3 |

The `TRUE_EFFECT_*` constants below define the ground truth so we can later compare model estimates against it.

In [ ]:
# ----------------------------------------------------------------------------
# GROUND TRUTH -- the true synthetic treatment effect the platform must recover
# ----------------------------------------------------------------------------
TRUE_EFFECT_MODERATE = 0.45   # t/ha uplift under moderate water stress
TRUE_EFFECT_NONE     = 0.05   # t/ha under no/low stress
TRUE_EFFECT_SEVERE   = 0.00   # t/ha under severe drought
TRUE_EFFECT_SALINE   = -0.05  # t/ha under high salinity
WRONG_STAGE_MULT     = 0.30   # multiplier if applied at a non-responsive crop stage

rng = np.random.default_rng(RANDOM_SEED)

sites = [f"S{i+1}" for i in range(N_SITES)]
# each site has a base productivity and a climate profile
site_base_yield = {s: rng.uniform(3.5, 6.0) for s in sites}   # t/ha baseline
site_lat = {s: rng.uniform(-5, 20) for s in sites}
site_lon = {s: rng.uniform(95, 130) for s in sites}

varieties = ["Roma", "Heirloom", "Cherry", "Beefsteak"]
irrigations = ["drip", "furrow", "sprinkler", "rainfed"]
methods = ["foliar", "soil_drench", "fertigation"]
stages = ["vegetative", "flowering", "fruit_set", "ripening"]
responsive_stages = {"flowering", "fruit_set"}
textures = ["sandy_loam", "clay_loam", "loam", "silt_loam"]


def classify_stress(drought_index, ec_ds_m):
    """Map raw environment to a discrete stress regime (drives ground-truth effect)."""
    if ec_ds_m >= 4.0:
        return "high_salinity"
    if drought_index >= 0.70:
        return "severe_drought"
    if 0.35 <= drought_index < 0.70:
        return "moderate_stress"
    return "no_stress"


def true_effect_for(stress_regime, crop_stage):
    base = {
        "moderate_stress": TRUE_EFFECT_MODERATE,
        "no_stress": TRUE_EFFECT_NONE,
        "severe_drought": TRUE_EFFECT_SEVERE,
        "high_salinity": TRUE_EFFECT_SALINE,
    }[stress_regime]
    if crop_stage not in responsive_stages:
        base *= WRONG_STAGE_MULT
    return base


rows = []
for i in range(N_FIELDS):
    fid = f"F{i+1:03d}"
    site = rng.choice(sites)

    # --- soil ---
    ph = float(np.clip(rng.normal(6.3, 0.6), 4.5, 8.5))
    ec = float(np.clip(rng.gamma(2.0, 1.1), 0.2, 8.0))          # dS/m; salinity
    om = float(np.clip(rng.normal(2.4, 0.8), 0.3, 6.0))         # organic matter %
    cec = float(np.clip(rng.normal(15, 4), 4, 35))
    texture = rng.choice(textures)

    # --- weather (drives drought stress) ---
    drought = float(np.clip(rng.beta(2.2, 3.0), 0.02, 0.98))    # 0 wet .. 1 dry
    rainfall_pre = float(np.clip(rng.normal(180 - 120 * drought, 30), 10, 400))
    rainfall_post = float(np.clip(rng.normal(160 - 120 * drought, 30), 10, 400))
    temp_avg = float(np.clip(rng.normal(28 + 3 * drought, 1.5), 20, 38))
    temp_max = temp_avg + float(np.clip(rng.normal(6, 1.5), 2, 12))
    vpd = float(np.clip(rng.normal(1.2 + 1.8 * drought, 0.3), 0.4, 4.5))

    stress_regime = classify_stress(drought, ec)

    # --- treatment assignment (randomized ~50/50) ---
    treated = int(rng.random() < 0.5)
    stage = rng.choice(stages, p=[0.25, 0.30, 0.30, 0.15])
    method = rng.choice(methods)
    dose = float(rng.choice([0.0]) ) if not treated else float(rng.normal(2500, 200))
    app_date = datetime(2024, 3, 1) + timedelta(days=int(rng.integers(0, 60)))

    # --- baseline (pre-treatment) vegetation, correlated with soil/stress ---
    veg_vigor = (
        0.55
        + 0.05 * (om - 2.4)
        - 0.25 * drought
        - 0.03 * max(0.0, ec - 2.0)
        + rng.normal(0, 0.03)
    )
    veg_vigor = float(np.clip(veg_vigor, 0.15, 0.9))

    ndvi_pre = float(np.clip(veg_vigor + rng.normal(0, 0.02), 0.1, 0.95))
    ndre_pre = float(np.clip(0.6 * veg_vigor + rng.normal(0, 0.02), 0.05, 0.7))
    ndwi_pre = float(np.clip(0.4 - 0.3 * drought + rng.normal(0, 0.02), -0.3, 0.6))
    savi_pre = float(np.clip(ndvi_pre * 0.85 + rng.normal(0, 0.02), 0.05, 0.9))

    # --- ground-truth treatment effect for this field ---
    te = true_effect_for(stress_regime, stage) if treated else 0.0
    # small heterogeneity noise around the true effect
    te_realized = te + (rng.normal(0, 0.05) if treated else 0.0)

    # --- post-treatment vegetation: treated + moderate stress recovers NDRE/NDWI ---
    recovery = 0.0
    if treated and stress_regime == "moderate_stress" and stage in responsive_stages:
        recovery = rng.normal(0.06, 0.015)   # NDRE/NDWI recovery signal
    ndre_post = float(np.clip(ndre_pre + rng.normal(-0.01, 0.02) + recovery, 0.05, 0.75))
    ndwi_post = float(np.clip(ndwi_pre + rng.normal(-0.02, 0.02) + 1.2 * recovery, -0.35, 0.65))
    ndvi_post = float(np.clip(ndvi_pre + rng.normal(-0.01, 0.02) + 0.8 * recovery, 0.1, 0.97))
    savi_post = float(np.clip(savi_pre + rng.normal(-0.01, 0.02) + 0.7 * recovery, 0.05, 0.92))

    # --- Sentinel-1 radar (structure/moisture) & Sentinel-3 LST ---
    s1_vv = float(rng.normal(-8 - 4 * drought, 1.2))
    s1_vh = float(rng.normal(-14 - 4 * drought, 1.2))
    s1_ratio = s1_vv - s1_vh
    lst = float(np.clip(rng.normal(30 + 6 * drought, 2.0), 22, 48))

    # --- yield model (ground-truth data-generating process) ---
    base_yield = (
        site_base_yield[site]
        + 1.8 * (veg_vigor - 0.5)
        + 0.15 * (om - 2.4)
        - 1.6 * drought
        - 0.12 * max(0.0, ec - 2.0)
        - 0.02 * max(0.0, temp_max - 34)
    )
    yield_t_ha = float(np.clip(base_yield + te_realized + rng.normal(0, 0.25), 0.4, 12.0))
    marketable = float(np.clip(yield_t_ha * rng.uniform(0.82, 0.95), 0.3, 12.0))
    quality = float(np.clip(rng.normal(72 + 12 * (te_realized > 0.2), 6), 40, 98))

    # --- economics ---
    crop_price = float(rng.normal(320, 25))       # USD / t
    product_cost = float(0.0 if not treated else rng.normal(85, 8))
    app_cost = float(0.0 if not treated else rng.normal(35, 5))

    rows.append(dict(
        field_id=fid, site_id=site, variety=rng.choice(varieties),
        lat=site_lat[site] + rng.normal(0, 0.2), lon=site_lon[site] + rng.normal(0, 0.2),
        area_ha=float(np.clip(rng.normal(1.5, 0.5), 0.3, 5.0)),
        irrigation_type=rng.choice(irrigations),
        historical_yield_avg=float(np.clip(site_base_yield[site] + rng.normal(0, 0.4), 1.0, 11.0)),
        ph=ph, ec_ds_m=ec, organic_matter_pct=om, cec=cec, texture_class=texture,
        drought_index=drought, rainfall_pre=rainfall_pre, rainfall_post=rainfall_post,
        temp_avg_c=temp_avg, temp_max_c=temp_max, vpd_kpa=vpd,
        stress_regime=stress_regime,
        treatment_flag=treated, dose_ml_ha=dose, application_method=method,
        crop_stage=stage, application_date=app_date,
        ndvi_pre=ndvi_pre, ndre_pre=ndre_pre, ndwi_pre=ndwi_pre, savi_pre=savi_pre,
        ndvi_post=ndvi_post, ndre_post=ndre_post, ndwi_post=ndwi_post, savi_post=savi_post,
        s1_vv=s1_vv, s1_vh=s1_vh, s1_ratio=s1_ratio, lst=lst,
        yield_t_ha=yield_t_ha, marketable_yield_t_ha=marketable, quality_score=quality,
        crop_price_usd_t=crop_price, product_cost_usd=product_cost, application_cost_usd=app_cost,
        true_effect=te_realized,
    ))

gen = pd.DataFrame(rows)
print(f"Generated {len(gen)} synthetic fields.")
print(gen["stress_regime"].value_counts())
print("\nTreated vs control:")
print(gen["treatment_flag"].value_counts())

In [ ]:
# ----------------------------------------------------------------------------
# Write the generated data into the 7 normalized agronomic tables.
# The `true_effect` and `stress_regime` columns are ground-truth helpers; we keep
# them out of the modeling tables and stash them in a separate ground_truth table.
# ----------------------------------------------------------------------------

# formulations (single product row + a control placeholder)
formulations_df = pd.DataFrame([
    dict(product_id="BIOXA", product_name="BioX-A", active_ingredient="seaweed_extract_ascophyllum",
         concentration_pct=12.0, description="Synthetic tropical seaweed biostimulant (POC)"),
])

fields_df = gen[["field_id", "site_id", "lat", "lon", "area_ha", "irrigation_type",
                 "historical_yield_avg", "variety"]].copy()
fields_df["crop"] = CROP

soil_df = gen[["field_id", "ph", "ec_ds_m", "organic_matter_pct", "texture_class", "cec"]].copy()

treatments_df = gen[["field_id", "treatment_flag", "dose_ml_ha", "application_method",
                     "crop_stage", "application_date"]].copy()
treatments_df["product_id"] = np.where(gen["treatment_flag"] == 1, "BIOXA", None)

# weather -> long format (pre/post)
weather_pre = gen[["field_id", "rainfall_pre", "temp_avg_c", "temp_max_c", "vpd_kpa", "drought_index"]].copy()
weather_pre = weather_pre.rename(columns={"rainfall_pre": "rainfall_mm"})
weather_pre["period"] = "pre"
weather_post = gen[["field_id", "rainfall_post", "temp_avg_c", "temp_max_c", "vpd_kpa", "drought_index"]].copy()
weather_post = weather_post.rename(columns={"rainfall_post": "rainfall_mm"})
weather_post["period"] = "post"
weather_df = pd.concat([weather_pre, weather_post], ignore_index=True)

# satellite -> long format (pre/post)
def sat_frame(period):
    d = gen[["field_id",
             f"ndvi_{period}", f"ndre_{period}", f"ndwi_{period}", f"savi_{period}",
             "s1_vv", "s1_vh", "s1_ratio", "lst"]].copy()
    d.columns = ["field_id", "ndvi", "ndre", "ndwi", "savi",
                 "sentinel1_vv", "sentinel1_vh", "sentinel1_vv_vh", "sentinel3_lst"]
    d["period"] = period
    return d
satellite_df = pd.concat([sat_frame("pre"), sat_frame("post")], ignore_index=True)

outcomes_df = gen[["field_id", "yield_t_ha", "marketable_yield_t_ha", "quality_score",
                   "product_cost_usd", "application_cost_usd", "crop_price_usd_t"]].copy()

# ground truth kept separately (for evaluation only -- NOT a model feature)
ground_truth_df = gen[["field_id", "stress_regime", "true_effect"]].copy()

# Write (append into freshly-created empty tables)
formulations_df.to_sql("formulations", engine, if_exists="append", index=False)
fields_df.to_sql("fields", engine, if_exists="append", index=False)
soil_df.to_sql("soil", engine, if_exists="append", index=False)
treatments_df.to_sql("treatments", engine, if_exists="append", index=False)
weather_df.to_sql("weather", engine, if_exists="append", index=False)
satellite_df.to_sql("satellite_indices", engine, if_exists="append", index=False)
outcomes_df.to_sql("outcomes", engine, if_exists="append", index=False)
ground_truth_df.to_sql("ground_truth", engine, if_exists="replace", index=False)

print("Wrote agronomic tables to Postgres:")
for t in ["formulations", "fields", "soil", "treatments", "weather", "satellite_indices", "outcomes", "ground_truth"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"].iloc[0]
    print(f"  {t:20s}: {n} rows")

## 4. Read back from Postgres & build the master analytical frame

Everything below reads *from Postgres* (not the in-memory generator), proving the data-engineering round-trip. We reshape the long pre/post weather and satellite tables into wide columns.

In [ ]:
# Read normalized tables back from Postgres
fields = pd.read_sql("SELECT * FROM fields", engine)
soil = pd.read_sql("SELECT * FROM soil", engine)
treatments = pd.read_sql("SELECT * FROM treatments", engine)
weather = pd.read_sql("SELECT * FROM weather", engine)
satellite = pd.read_sql("SELECT * FROM satellite_indices", engine)
outcomes = pd.read_sql("SELECT * FROM outcomes", engine)
ground_truth = pd.read_sql("SELECT * FROM ground_truth", engine)

# Pivot weather pre/post to wide
w = weather.pivot_table(index="field_id", columns="period",
                        values=["rainfall_mm", "temp_avg_c", "temp_max_c", "vpd_kpa", "drought_index"])
w.columns = [f"{a}_{b}" for a, b in w.columns]
w = w.reset_index()

# Pivot satellite pre/post to wide
sat = satellite.pivot_table(index="field_id", columns="period",
                            values=["ndvi", "ndre", "ndwi", "savi",
                                    "sentinel1_vv", "sentinel1_vh", "sentinel1_vv_vh", "sentinel3_lst"])
sat.columns = [f"{a}_{b}" for a, b in sat.columns]
sat = sat.reset_index()

# Master frame
master = (fields
          .merge(soil, on="field_id")
          .merge(treatments, on="field_id")
          .merge(w, on="field_id")
          .merge(sat, on="field_id")
          .merge(outcomes, on="field_id")
          .merge(ground_truth, on="field_id"))

# numeric casting (Postgres NUMERIC -> object sometimes)
num_cols = master.select_dtypes(include=["object"]).columns.difference(
    ["field_id", "site_id", "crop", "variety", "irrigation_type", "texture_class",
     "product_id", "application_method", "crop_stage", "stress_regime"])
for c in num_cols:
    master[c] = pd.to_numeric(master[c], errors="ignore")
for c in master.columns:
    if master[c].dtype == object and c not in [
        "field_id", "site_id", "crop", "variety", "irrigation_type", "texture_class",
        "product_id", "application_method", "crop_stage", "stress_regime", "application_date"]:
        master[c] = pd.to_numeric(master[c], errors="coerce")

print("Master frame:", master.shape)
master.head(3)

In [ ]:
# ---- EDA: yield by treatment group and stress regime ----
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=master, x="stress_regime", y="yield_t_ha", hue="treatment_flag",
            order=["no_stress", "moderate_stress", "severe_drought", "high_salinity"], ax=axes[0])
axes[0].set_title("Yield by stress regime: control (0) vs BioX-A (1)")
axes[0].set_xlabel("stress regime"); axes[0].set_ylabel("yield (t/ha)")
axes[0].tick_params(axis="x", rotation=20)

# raw group means table
grp = (master.groupby(["stress_regime", "treatment_flag"])["yield_t_ha"]
       .mean().unstack().reindex(["no_stress", "moderate_stress", "severe_drought", "high_salinity"]))
grp["naive_diff"] = grp[1] - grp[0]
sns.heatmap(grp[[0, 1]].astype(float), annot=True, fmt=".2f", cmap="YlGn", ax=axes[1])
axes[1].set_title("Mean yield (t/ha) by regime x treatment")
plt.tight_layout(); plt.show()

print("Naive (unadjusted) treated-minus-control by stress regime:")
print(grp["naive_diff"].round(3))
print("\nReminder -- embedded ground truth: moderate=+0.45, none=+0.05, severe=0.0, saline=-0.05")

In [ ]:
# ---- EDA: treatment balance across sites + stress/moisture distributions ----
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

bal = master.groupby(["site_id", "treatment_flag"]).size().unstack(fill_value=0)
bal.plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Treatment balance by site"); axes[0].set_ylabel("n fields")

sns.histplot(master, x="drought_index", hue="treatment_flag", bins=25, kde=True, ax=axes[1])
axes[1].set_title("Drought index distribution")

sns.scatterplot(data=master, x="drought_index", y="yield_t_ha", hue="treatment_flag",
                alpha=0.5, ax=axes[2])
axes[2].set_title("Yield vs drought index")
plt.tight_layout(); plt.show()

In [ ]:
# ---- EDA: interactive pre/post NDRE & NDWI response (treated vs control, moderate stress) ----
import plotly.express as px
import plotly.graph_objects as go

mod = master[master["stress_regime"] == "moderate_stress"].copy()
resp = pd.DataFrame({
    "group": ["control", "control", "BioX-A", "BioX-A"],
    "period": ["pre", "post", "pre", "post"],
    "NDRE": [
        mod.loc[mod.treatment_flag == 0, "ndre_pre"].mean(),
        mod.loc[mod.treatment_flag == 0, "ndre_post"].mean(),
        mod.loc[mod.treatment_flag == 1, "ndre_pre"].mean(),
        mod.loc[mod.treatment_flag == 1, "ndre_post"].mean(),
    ],
    "NDWI": [
        mod.loc[mod.treatment_flag == 0, "ndwi_pre"].mean(),
        mod.loc[mod.treatment_flag == 0, "ndwi_post"].mean(),
        mod.loc[mod.treatment_flag == 1, "ndwi_pre"].mean(),
        mod.loc[mod.treatment_flag == 1, "ndwi_post"].mean(),
    ],
})

fig = px.line(resp, x="period", y="NDRE", color="group", markers=True,
              title="Moderate-stress fields: mean NDRE pre vs post (difference-in-differences signal)")
fig.show()
fig2 = px.line(resp, x="period", y="NDWI", color="group", markers=True,
               title="Moderate-stress fields: mean NDWI pre vs post")
fig2.show()

## 5. Feature engineering → `feature_store`

Engineer biostimulant-relevant features and persist them to a `feature_store` table in Postgres (this is the single table the modeling and experiment layers read from).

Key features:
- **Delta / difference-in-differences**: `ndre_delta`, `ndwi_delta`, `ndvi_delta` (post − pre)
- **Stress-window indicator**: binary moderate-water-stress flag at application
- **Rainfall anomaly**: deviation from site-mean rainfall
- **Treatment × stress interaction**
- **Vegetation index AUC** (trapezoidal pre→post)
- **Historical productivity class** (site-quantile based)
- **Crop-stage appropriateness** flag

In [ ]:
fs = master.copy()

# --- vegetation delta / difference-in-differences features ---
fs["ndre_delta"] = fs["ndre_post"] - fs["ndre_pre"]
fs["ndwi_delta"] = fs["ndwi_post"] - fs["ndwi_pre"]
fs["ndvi_delta"] = fs["ndvi_post"] - fs["ndvi_pre"]
fs["savi_delta"] = fs["savi_post"] - fs["savi_pre"]

# --- vegetation index AUC (trapezoid over 2 points = mean of pre/post) ---
fs["ndvi_auc"] = (fs["ndvi_pre"] + fs["ndvi_post"]) / 2.0
fs["ndre_auc"] = (fs["ndre_pre"] + fs["ndre_post"]) / 2.0

# --- NDVI growth slope (post-pre over unit interval) ---
fs["ndvi_slope"] = fs["ndvi_delta"]

# --- stress-window indicator (moderate water stress at application) ---
fs["moderate_stress_flag"] = ((fs["drought_index_pre"] >= 0.35) &
                              (fs["drought_index_pre"] < 0.70) &
                              (fs["ec_ds_m"] < 4.0)).astype(int)
fs["severe_drought_flag"] = (fs["drought_index_pre"] >= 0.70).astype(int)
fs["high_salinity_flag"] = (fs["ec_ds_m"] >= 4.0).astype(int)

# --- rainfall anomaly vs site mean ---
site_rain = fs.groupby("site_id")["rainfall_mm_pre"].transform("mean")
fs["rainfall_anomaly"] = fs["rainfall_mm_pre"] - site_rain

# --- crop-stage appropriateness ---
fs["stage_appropriate"] = fs["crop_stage"].isin(["flowering", "fruit_set"]).astype(int)

# --- treatment x stress interaction ---
fs["treat_x_modstress"] = fs["treatment_flag"] * fs["moderate_stress_flag"]
fs["treat_x_drought"] = fs["treatment_flag"] * fs["drought_index_pre"]

# --- historical productivity class (site-quantile) ---
fs["productivity_class"] = pd.qcut(fs["historical_yield_avg"], 3,
                                   labels=["low", "medium", "high"]).astype(str)

# --- pre-treatment NDRE below site median (a targeting signal) ---
site_ndre_med = fs.groupby("site_id")["ndre_pre"].transform("median")
fs["ndre_below_median"] = (fs["ndre_pre"] < site_ndre_med).astype(int)

# --- ROI-relevant economics (uplift filled later by the uplift model) ---
fs["total_cost_usd"] = fs["product_cost_usd"] + fs["application_cost_usd"]

feature_cols = [
    "field_id", "site_id", "crop", "variety", "irrigation_type", "texture_class",
    "crop_stage", "application_method", "productivity_class",
    # soil
    "ph", "ec_ds_m", "organic_matter_pct", "cec",
    # weather
    "drought_index_pre", "vpd_kpa_pre", "temp_max_c_pre", "rainfall_mm_pre",
    "rainfall_anomaly",
    # satellite pre + deltas + auc
    "ndvi_pre", "ndre_pre", "ndwi_pre", "savi_pre",
    "ndre_delta", "ndwi_delta", "ndvi_delta", "savi_delta",
    "ndvi_auc", "ndre_auc", "ndvi_slope",
    "sentinel1_vv_pre", "sentinel1_vh_pre", "sentinel1_vv_vh_pre", "sentinel3_lst_pre",
    # flags & interactions
    "moderate_stress_flag", "severe_drought_flag", "high_salinity_flag",
    "stage_appropriate", "ndre_below_median",
    "treatment_flag", "treat_x_modstress", "treat_x_drought", "dose_ml_ha",
    # economics
    "product_cost_usd", "application_cost_usd", "total_cost_usd", "crop_price_usd_t",
    # targets & ground truth
    "yield_t_ha", "marketable_yield_t_ha", "quality_score",
    "stress_regime", "true_effect",
]
feature_store = fs[feature_cols].copy()

# numeric NaN check
n_nan = feature_store.select_dtypes("number").isna().sum().sum()
print("Feature store shape:", feature_store.shape, "| numeric NaNs:", int(n_nan))

# persist
feature_store.to_sql("feature_store", engine, if_exists="replace", index=False)
print("Wrote feature_store to Postgres.")
feature_store.head(3)

## 6. Core models (reusable functions)

Three model builders, each returning a fitted object plus metrics/artifacts so the experiments layer can wrap them:

1. `train_yield_model(...)` — yield prediction (linear baseline, Random Forest, XGBoost) with **leave-site-out** CV
2. `estimate_uplift(...)` — causal treatment effect (statsmodels OLS causal + optional EconML CausalForest), CATE by stress segment, compared to known ground truth
3. `recommend_fields(...)` — deterministic recommendation engine with ROI

We first define the feature matrix builder shared by all models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Named feature sets so experiments can vary inputs
CATEG_FEATURES = ["irrigation_type", "texture_class", "crop_stage",
                  "application_method", "productivity_class"]

NUMERIC_BASE = ["ph", "ec_ds_m", "organic_matter_pct", "cec",
                "drought_index_pre", "vpd_kpa_pre", "temp_max_c_pre",
                "rainfall_mm_pre"]

NUMERIC_SATELLITE = ["ndvi_pre", "ndre_pre", "ndwi_pre", "savi_pre",
                     "ndre_delta", "ndwi_delta", "ndvi_delta", "savi_delta",
                     "ndvi_auc", "ndre_auc", "ndvi_slope",
                     "sentinel1_vv_pre", "sentinel1_vh_pre", "sentinel1_vv_vh_pre",
                     "sentinel3_lst_pre"]

NUMERIC_INTERACTIONS = ["moderate_stress_flag", "severe_drought_flag",
                        "high_salinity_flag", "stage_appropriate",
                        "ndre_below_median", "treat_x_modstress", "treat_x_drought"]

FEATURE_SETS = {
    "baseline":        (NUMERIC_BASE, CATEG_FEATURES),
    "with_satellite":  (NUMERIC_BASE + NUMERIC_SATELLITE, CATEG_FEATURES),
    "full":            (NUMERIC_BASE + ["rainfall_anomaly"]
                        + NUMERIC_SATELLITE + NUMERIC_INTERACTIONS,
                        CATEG_FEATURES),
}


def build_preprocessor(numeric, categorical):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), numeric),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])


def get_Xy(df, feature_set="full", include_treatment=True):
    numeric, categorical = FEATURE_SETS[feature_set]
    numeric = list(dict.fromkeys(numeric))  # dedupe
    cols = numeric + categorical
    if include_treatment:
        cols = cols + ["treatment_flag", "dose_ml_ha"]
        numeric = numeric + ["treatment_flag", "dose_ml_ha"]
    X = df[cols].copy()
    y = df["yield_t_ha"].astype(float)
    groups = df["site_id"]
    return X, y, groups, numeric, categorical

print("Feature sets available:", list(FEATURE_SETS.keys()))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False


def _make_model(model_type, params):
    params = params or {}
    if model_type == "linear_regression":
        return LinearRegression()
    if model_type == "random_forest":
        return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                                     n_estimators=params.get("n_estimators", 300),
                                     max_depth=params.get("max_depth", None),
                                     min_samples_leaf=params.get("min_samples_leaf", 2))
    if model_type == "xgboost" and HAS_XGB:
        return XGBRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                            n_estimators=params.get("n_estimators", 400),
                            max_depth=params.get("max_depth", 5),
                            learning_rate=params.get("learning_rate", 0.05),
                            subsample=0.9, colsample_bytree=0.9)
    if model_type == "lightgbm" and HAS_LGBM:
        return LGBMRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                             n_estimators=params.get("n_estimators", 400),
                             max_depth=params.get("max_depth", -1),
                             learning_rate=params.get("learning_rate", 0.05))
    # fallback
    return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)


def train_yield_model(df, model_type="random_forest", feature_set="full",
                      params=None, cv_strategy="leave_site_out"):
    """Fit a yield model on the full data and return the fitted pipeline.
    (Cross-validated metrics are produced separately by evaluate_yield.)"""
    X, y, groups, numeric, categorical = get_Xy(df, feature_set, include_treatment=True)
    pipe = Pipeline([
        ("prep", build_preprocessor(numeric, categorical)),
        ("model", _make_model(model_type, params)),
    ])
    pipe.fit(X, y)
    meta = {"model_type": model_type, "feature_set": feature_set,
            "params": params or {}, "cv_strategy": cv_strategy,
            "n_train": len(y), "numeric": numeric, "categorical": categorical}
    return pipe, meta


def evaluate_yield(df, model_type="random_forest", feature_set="full",
                   params=None, cv_strategy="leave_site_out"):
    """Cross-validated yield metrics. Returns per-fold and aggregate RMSE/MAE/R2."""
    X, y, groups, numeric, categorical = get_Xy(df, feature_set, include_treatment=True)

    if cv_strategy == "leave_site_out":
        splitter = LeaveOneGroupOut()
        split_iter = splitter.split(X, y, groups)
    elif cv_strategy == "random_80_20":
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
        split_iter = splitter.split(X, y, groups)
    else:
        raise ValueError(cv_strategy)

    fold_rows, preds_all, truth_all = [], [], []
    for k, (tr, te) in enumerate(split_iter):
        pipe = Pipeline([("prep", build_preprocessor(numeric, categorical)),
                         ("model", _make_model(model_type, params))])
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[te])
        fold_rows.append({
            "fold": k,
            "rmse": float(np.sqrt(mean_squared_error(y.iloc[te], pred))),
            "mae": float(mean_absolute_error(y.iloc[te], pred)),
            "r2": float(r2_score(y.iloc[te], pred)),
            "held_site": groups.iloc[te].iloc[0] if cv_strategy == "leave_site_out" else "random",
        })
        preds_all.append(pred); truth_all.append(y.iloc[te].values)

    fold_df = pd.DataFrame(fold_rows)
    preds_all = np.concatenate(preds_all); truth_all = np.concatenate(truth_all)
    agg = {
        "rmse": float(np.sqrt(mean_squared_error(truth_all, preds_all))),
        "mae": float(mean_absolute_error(truth_all, preds_all)),
        "r2": float(r2_score(truth_all, preds_all)),
    }
    return {"fold_metrics": fold_df, "aggregate": agg,
            "predictions": preds_all, "truth": truth_all}


# quick smoke test
_yield_eval = evaluate_yield(feature_store, "random_forest", "full")
print("Leave-site-out RF yield metrics (aggregate):")
print("  ", {k: round(v, 3) for k, v in _yield_eval["aggregate"].items()})
_yield_eval["fold_metrics"].round(3)

In [ ]:
import statsmodels.formula.api as smf

try:
    from econml.dml import CausalForestDML
    from sklearn.ensemble import RandomForestRegressor as _RF
    HAS_ECONML = True
except Exception:
    HAS_ECONML = False

# segments we care about for ground-truth comparison
STRESS_SEGMENTS = ["moderate_stress", "no_stress", "severe_drought", "high_salinity"]
TRUE_EFFECT_BY_SEGMENT = {
    "moderate_stress": TRUE_EFFECT_MODERATE,
    "no_stress": TRUE_EFFECT_NONE,
    "severe_drought": TRUE_EFFECT_SEVERE,
    "high_salinity": TRUE_EFFECT_SALINE,
}


def estimate_uplift(df, method="ols_causal", feature_set="full", params=None):
    """Estimate per-field treatment effect (CATE) of BioX-A vs control.

    Returns a dict with:
      - 'cate': np.array per-field estimated uplift (t/ha)
      - 'ate': overall average treatment effect
      - 'segment_estimates': dict stress_regime -> estimated uplift
      - 'model': fitted object (for reproducibility)
    """
    d = df.copy()
    d["T"] = d["treatment_flag"].astype(int)

    if method == "ols_causal":
        # DiD-style OLS with treatment x stress interactions + covariates
        formula = ("yield_t_ha ~ T + moderate_stress_flag + severe_drought_flag "
                   "+ high_salinity_flag + stage_appropriate "
                   "+ T:moderate_stress_flag + T:severe_drought_flag "
                   "+ T:high_salinity_flag + T:stage_appropriate "
                   "+ ndre_delta + ndwi_delta + drought_index_pre + ec_ds_m "
                   "+ organic_matter_pct + ndre_pre")
        model = smf.ols(formula, data=d).fit()

        # predict counterfactual: uplift = yhat(T=1) - yhat(T=0)
        d1 = d.copy(); d1["T"] = 1
        d0 = d.copy(); d0["T"] = 0
        cate = model.predict(d1).values - model.predict(d0).values
        model_obj = model

    elif method == "causal_forest" and HAS_ECONML:
        numeric, categorical = FEATURE_SETS[feature_set]
        Xc = pd.get_dummies(d[numeric + categorical], drop_first=True).astype(float)
        Xc = Xc.fillna(Xc.median(numeric_only=True))
        est = CausalForestDML(
            model_y=_RF(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1),
            model_t=_RF(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1),
            n_estimators=(params or {}).get("n_estimators", 400),
            random_state=RANDOM_SEED,
        )
        est.fit(Y=d["yield_t_ha"].values, T=d["T"].values, X=Xc.values)
        cate = est.effect(Xc.values)
        model_obj = est
    else:
        # graceful fallback to OLS if EconML missing
        return estimate_uplift(df, method="ols_causal", feature_set=feature_set, params=params)

    d["_cate"] = cate
    segment_estimates = {
        seg: float(d.loc[d["stress_regime"] == seg, "_cate"].mean())
        for seg in STRESS_SEGMENTS if (d["stress_regime"] == seg).any()
    }
    return {
        "cate": cate,
        "ate": float(np.mean(cate)),
        "segment_estimates": segment_estimates,
        "method": method if (method != "causal_forest" or HAS_ECONML) else "ols_causal",
        "model": model_obj,
        "field_ids": d["field_id"].values,
    }


def uplift_metrics(uplift_result, df):
    """Compare estimated uplift to the KNOWN synthetic ground truth."""
    seg_est = uplift_result["segment_estimates"]
    rows = []
    for seg, true_val in TRUE_EFFECT_BY_SEGMENT.items():
        est = seg_est.get(seg, np.nan)
        rows.append({"segment": seg, "true_uplift": true_val,
                     "est_uplift": est, "bias": est - true_val})
    seg_df = pd.DataFrame(rows)

    # per-field truth vs estimate (treated fields have realized true_effect)
    d = df.copy()
    d["est_cate"] = uplift_result["cate"]
    treated = d[d["treatment_flag"] == 1]
    field_bias = float((treated["est_cate"] - treated["true_effect"]).abs().mean())

    metrics = {
        "ate_estimate": uplift_result["ate"],
        "treatment_effect_error_mae": field_bias,
        "cate_error_moderate": float(seg_df.loc[seg_df.segment == "moderate_stress", "bias"].abs().iloc[0]),
        "true_uplift_moderate": TRUE_EFFECT_MODERATE,
        "est_uplift_moderate": float(seg_df.loc[seg_df.segment == "moderate_stress", "est_uplift"].iloc[0]),
    }
    return metrics, seg_df


# smoke test
_upl = estimate_uplift(feature_store, "ols_causal", "full")
_m, _seg = uplift_metrics(_upl, feature_store)
print("Uplift segment estimates vs ground truth:")
print(_seg.round(3).to_string(index=False))
print("\nModerate-stress: true = +{:.2f} t/ha | estimated = {:+.2f} t/ha".format(
    _m["true_uplift_moderate"], _m["est_uplift_moderate"]))

In [ ]:
# Default recommendation rule set (deterministic, auditable)
DEFAULT_RULES = {
    "uplift_threshold": 0.0,        # predicted uplift must be positive
    "roi_threshold": 1.5,           # ROI must exceed this multiple
    "confidence_lower_bound": -0.05,  # lower CI bound on uplift (t/ha) must exceed this
    "allowed_crop_stages": ["flowering", "fruit_set"],
    "max_soil_ec": 4.0,             # exclude severely saline fields
}


def recommend_fields(df, uplift_result, rules=None, ci_halfwidth=0.12):
    """Deterministic field-level recommendation with ROI, confidence, drivers.

    ROI = (uplift_t_ha * crop_price) / total_cost   (per hectare)
    Confidence bucket derived from |uplift| relative to CI half-width.
    """
    rules = {**DEFAULT_RULES, **(rules or {})}
    out = df.copy()
    out["pred_uplift_t_ha"] = uplift_result["cate"]

    # simple symmetric CI (POC): halfwidth shrinks with clearer signal
    out["uplift_ci_low"] = out["pred_uplift_t_ha"] - ci_halfwidth
    out["uplift_ci_high"] = out["pred_uplift_t_ha"] + ci_halfwidth

    # economics
    total_cost = out["total_cost_usd"].replace(0, np.nan)
    # for control fields (cost 0) assume standard-dose cost for the what-if
    std_cost = out.loc[out["treatment_flag"] == 1, "total_cost_usd"].median()
    total_cost = total_cost.fillna(std_cost if not np.isnan(std_cost) else 120.0)
    out["expected_gain_usd"] = out["pred_uplift_t_ha"] * out["crop_price_usd_t"]
    out["roi"] = out["expected_gain_usd"] / total_cost
    out["uplift_pct"] = 100.0 * out["pred_uplift_t_ha"] / out["yield_t_ha"].clip(lower=0.1)

    # confidence bucket
    def conf(row):
        signal = row["pred_uplift_t_ha"] / (ci_halfwidth + 1e-9)
        if signal >= 2.0:
            return "high"
        if signal >= 1.0:
            return "medium-high"
        if signal >= 0.4:
            return "medium"
        return "low"
    out["confidence"] = out.apply(conf, axis=1)

    # deterministic rule checks
    c_uplift = out["pred_uplift_t_ha"] > rules["uplift_threshold"]
    c_roi = out["roi"] >= rules["roi_threshold"]
    c_ci = out["uplift_ci_low"] >= rules["confidence_lower_bound"]
    c_stage = out["crop_stage"].isin(rules["allowed_crop_stages"])
    c_ec = out["ec_ds_m"] <= rules["max_soil_ec"]

    out["recommend"] = (c_uplift & c_roi & c_ci & c_stage & c_ec)

    # human-readable drivers
    def drivers(row, i):
        d = []
        d.append(f"pred uplift {row['pred_uplift_t_ha']:+.2f} t/ha")
        d.append(f"ROI {row['roi']:.1f}x")
        if row["moderate_stress_flag"] == 1: d.append("moderate water stress")
        if row["ndwi_delta"] > 0: d.append("positive NDWI recovery")
        if row["ndre_delta"] > 0: d.append("positive NDRE recovery")
        if row["ndre_below_median"] == 1: d.append("below-median pre-NDRE")
        if row["ec_ds_m"] > rules["max_soil_ec"]: d.append("EXCLUDED: high salinity")
        if not c_stage.iloc[i]: d.append(f"EXCLUDED: stage={row['crop_stage']}")
        return "; ".join(d)
    out = out.reset_index(drop=True)
    out["key_drivers"] = [drivers(out.iloc[i], i) for i in range(len(out))]

    cols = ["field_id", "site_id", "stress_regime", "crop_stage", "yield_t_ha",
            "pred_uplift_t_ha", "uplift_pct", "uplift_ci_low", "uplift_ci_high",
            "roi", "confidence", "recommend", "key_drivers", "true_effect"]
    return out[cols], rules


def recommendation_metrics(rec_df):
    """Precision/recall of recommendation vs a ground-truth 'should treat' label.
    Ground truth: a field truly benefits if realized true_effect > 0.10 t/ha."""
    truth_benefit = rec_df["true_effect"] > 0.10
    pred = rec_df["recommend"]
    tp = int((pred & truth_benefit).sum())
    fp = int((pred & ~truth_benefit).sum())
    fn = int((~pred & truth_benefit).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return {"recommendation_precision": precision, "recommendation_recall": recall,
            "n_recommended": int(pred.sum())}


# smoke test
_rec, _rules = recommend_fields(feature_store, _upl)
_rm = recommendation_metrics(_rec)
print(f"Recommended {_rm['n_recommended']} of {len(_rec)} fields.")
print(f"Precision={_rm['recommendation_precision']:.2f}  Recall={_rm['recommendation_recall']:.2f}")
_rec[_rec["recommend"]].head(5)[["field_id", "stress_regime", "pred_uplift_t_ha", "roi", "confidence"]]

# Experiments Management Layer

The rest of the notebook builds the experiments management layer that wraps the models above.

**Hierarchy:** `Project → Experiment → Run → Evaluation → Metrics`

- **Project** — top-level container (e.g., "BioX-A Tomato Tropical Asia"). Supports many projects.
- **Experiment** — a hypothesis within a project.
- **Run** — one execution producing a *trained model* (parent MLflow run).
- **Evaluation** — a *first-class* scoring of that model against a scenario (nested MLflow run). Metrics attach here.

## 7. `biox_experiments` schema (12 tables + 2 views)

In [ ]:
EXP_DDL = f"""
CREATE SCHEMA IF NOT EXISTS {EXP_SCHEMA};

DROP TABLE IF EXISTS {EXP_SCHEMA}.experiment_comparisons CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.field_subsets CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.recommendation_rules CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.evaluation_artifacts CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.evaluation_metrics CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.evaluations CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.run_artifacts CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.run_parameters CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.experiment_runs CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.experiments CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.treatment_configs CASCADE;
DROP TABLE IF EXISTS {EXP_SCHEMA}.projects CASCADE;

CREATE TABLE {EXP_SCHEMA}.projects (
    project_id   BIGSERIAL PRIMARY KEY,
    name         TEXT NOT NULL,
    description  TEXT,
    crop         TEXT,
    product      TEXT,
    region       TEXT,
    created_by   TEXT,
    created_at   TIMESTAMPTZ DEFAULT now(),
    status       TEXT DEFAULT 'active'
);

CREATE TABLE {EXP_SCHEMA}.treatment_configs (
    config_id          BIGSERIAL PRIMARY KEY,
    product            TEXT,
    dose_ml_ha         NUMERIC(8,2),
    application_method TEXT,
    crop_stage         TEXT,
    stress_scenario    TEXT,
    description        TEXT
);

CREATE TABLE {EXP_SCHEMA}.experiments (
    experiment_id BIGSERIAL PRIMARY KEY,
    project_id    BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.projects(project_id) ON DELETE CASCADE,
    name          TEXT NOT NULL,
    description   TEXT,
    hypothesis    TEXT,
    model_family  TEXT,
    created_by    TEXT,
    created_at    TIMESTAMPTZ DEFAULT now(),
    status        TEXT DEFAULT 'draft'
);

CREATE TABLE {EXP_SCHEMA}.experiment_runs (
    run_id                 BIGSERIAL PRIMARY KEY,
    experiment_id          BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.experiments(experiment_id) ON DELETE CASCADE,
    mlflow_run_id          TEXT,
    mlflow_experiment_name TEXT,
    model_type             TEXT,
    start_time             TIMESTAMPTZ,
    end_time               TIMESTAMPTZ,
    status                 TEXT DEFAULT 'completed',
    git_commit             TEXT,
    notes                  TEXT
);

CREATE TABLE {EXP_SCHEMA}.run_parameters (
    param_id    BIGSERIAL PRIMARY KEY,
    run_id      BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.experiment_runs(run_id) ON DELETE CASCADE,
    param_name  TEXT NOT NULL,
    param_value TEXT,
    param_type  TEXT
);

CREATE TABLE {EXP_SCHEMA}.run_artifacts (
    artifact_id   BIGSERIAL PRIMARY KEY,
    run_id        BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.experiment_runs(run_id) ON DELETE CASCADE,
    artifact_name TEXT,
    artifact_path TEXT,
    artifact_type TEXT
);

CREATE TABLE {EXP_SCHEMA}.evaluations (
    evaluation_id       BIGSERIAL PRIMARY KEY,
    run_id              BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.experiment_runs(run_id) ON DELETE CASCADE,
    mlflow_run_id       TEXT,
    eval_scenario       TEXT NOT NULL,
    cv_strategy         TEXT,
    treatment_config_id BIGINT REFERENCES {EXP_SCHEMA}.treatment_configs(config_id),
    status              TEXT DEFAULT 'completed',
    created_at          TIMESTAMPTZ DEFAULT now(),
    notes               TEXT
);

CREATE TABLE {EXP_SCHEMA}.evaluation_metrics (
    metric_id     BIGSERIAL PRIMARY KEY,
    evaluation_id BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.evaluations(evaluation_id) ON DELETE CASCADE,
    metric_name   TEXT NOT NULL,
    metric_value  DOUBLE PRECISION,
    fold          INTEGER,
    recorded_at   TIMESTAMPTZ DEFAULT now()
);

CREATE TABLE {EXP_SCHEMA}.evaluation_artifacts (
    artifact_id   BIGSERIAL PRIMARY KEY,
    evaluation_id BIGINT NOT NULL REFERENCES {EXP_SCHEMA}.evaluations(evaluation_id) ON DELETE CASCADE,
    artifact_name TEXT,
    artifact_path TEXT,
    artifact_type TEXT
);

CREATE TABLE {EXP_SCHEMA}.recommendation_rules (
    rule_id                BIGSERIAL PRIMARY KEY,
    evaluation_id          BIGINT REFERENCES {EXP_SCHEMA}.evaluations(evaluation_id) ON DELETE CASCADE,
    uplift_threshold       DOUBLE PRECISION,
    roi_threshold          DOUBLE PRECISION,
    confidence_lower_bound DOUBLE PRECISION,
    allowed_crop_stages    JSONB,
    max_soil_ec            DOUBLE PRECISION,
    description            TEXT
);

CREATE TABLE {EXP_SCHEMA}.field_subsets (
    subset_id           BIGSERIAL PRIMARY KEY,
    evaluation_id       BIGINT REFERENCES {EXP_SCHEMA}.evaluations(evaluation_id) ON DELETE CASCADE,
    site_ids            JSONB,
    stress_level_filter TEXT,
    crop_stage_filter   TEXT,
    n_fields            INTEGER
);

CREATE TABLE {EXP_SCHEMA}.experiment_comparisons (
    comparison_id  BIGSERIAL PRIMARY KEY,
    name           TEXT,
    scope          TEXT,            -- 'run' | 'evaluation'
    evaluation_ids JSONB,
    run_ids        JSONB,
    created_by     TEXT,
    notes          TEXT,
    created_at     TIMESTAMPTZ DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_exp_project ON {EXP_SCHEMA}.experiments(project_id);
CREATE INDEX IF NOT EXISTS idx_run_exp ON {EXP_SCHEMA}.experiment_runs(experiment_id);
CREATE INDEX IF NOT EXISTS idx_eval_run ON {EXP_SCHEMA}.evaluations(run_id);
CREATE INDEX IF NOT EXISTS idx_metric_eval ON {EXP_SCHEMA}.evaluation_metrics(evaluation_id);
CREATE INDEX IF NOT EXISTS idx_param_run ON {EXP_SCHEMA}.run_parameters(run_id);
"""

with engine.begin() as conn:
    for stmt in [s for s in EXP_DDL.split(";") if s.strip()]:
        conn.execute(text(stmt))

print(f"Created schema '{EXP_SCHEMA}' with 12 tables + indexes.")

In [ ]:
# Two convenience views for the dashboard. Metrics are pivoted from the long
# evaluation_metrics table (aggregate rows have fold IS NULL).
VIEWS_DDL = f"""
CREATE OR REPLACE VIEW {EXP_SCHEMA}.v_evaluation_summary AS
SELECT
    p.project_id, p.name AS project_name,
    e.experiment_id, e.name AS experiment_name, e.model_family,
    r.run_id, r.model_type, r.mlflow_run_id AS run_mlflow_id,
    ev.evaluation_id, ev.eval_scenario, ev.cv_strategy, ev.created_at,
    MAX(CASE WHEN m.metric_name = 'rmse' AND m.fold IS NULL THEN m.metric_value END) AS rmse,
    MAX(CASE WHEN m.metric_name = 'mae' AND m.fold IS NULL THEN m.metric_value END) AS mae,
    MAX(CASE WHEN m.metric_name = 'r2' AND m.fold IS NULL THEN m.metric_value END) AS r2,
    MAX(CASE WHEN m.metric_name = 'uplift_bias' THEN m.metric_value END) AS uplift_bias,
    MAX(CASE WHEN m.metric_name = 'est_uplift_moderate' THEN m.metric_value END) AS est_uplift_moderate,
    MAX(CASE WHEN m.metric_name = 'treatment_effect_error_mae' THEN m.metric_value END) AS treatment_effect_error,
    MAX(CASE WHEN m.metric_name = 'recommendation_precision' THEN m.metric_value END) AS recommendation_precision,
    MAX(CASE WHEN m.metric_name = 'recommendation_recall' THEN m.metric_value END) AS recommendation_recall,
    MAX(CASE WHEN m.metric_name = 'roi_accuracy' THEN m.metric_value END) AS roi_accuracy
FROM {EXP_SCHEMA}.evaluations ev
JOIN {EXP_SCHEMA}.experiment_runs r ON r.run_id = ev.run_id
JOIN {EXP_SCHEMA}.experiments e ON e.experiment_id = r.experiment_id
JOIN {EXP_SCHEMA}.projects p ON p.project_id = e.project_id
LEFT JOIN {EXP_SCHEMA}.evaluation_metrics m ON m.evaluation_id = ev.evaluation_id
GROUP BY p.project_id, p.name, e.experiment_id, e.name, e.model_family,
         r.run_id, r.model_type, r.mlflow_run_id,
         ev.evaluation_id, ev.eval_scenario, ev.cv_strategy, ev.created_at;

CREATE OR REPLACE VIEW {EXP_SCHEMA}.v_run_summary AS
SELECT
    p.name AS project_name, e.name AS experiment_name,
    r.run_id, r.model_type, r.mlflow_run_id,
    COUNT(DISTINCT ev.evaluation_id) AS n_evaluations,
    MIN(vs.rmse) AS best_rmse,
    MAX(vs.r2) AS best_r2,
    MIN(ABS(vs.uplift_bias)) AS best_abs_uplift_bias
FROM {EXP_SCHEMA}.experiment_runs r
JOIN {EXP_SCHEMA}.experiments e ON e.experiment_id = r.experiment_id
JOIN {EXP_SCHEMA}.projects p ON p.project_id = e.project_id
LEFT JOIN {EXP_SCHEMA}.evaluations ev ON ev.run_id = r.run_id
LEFT JOIN {EXP_SCHEMA}.v_evaluation_summary vs ON vs.evaluation_id = ev.evaluation_id
GROUP BY p.name, e.name, r.run_id, r.model_type, r.mlflow_run_id;
"""

with engine.begin() as conn:
    for stmt in [s for s in VIEWS_DDL.split(";") if s.strip()]:
        conn.execute(text(stmt))

print("Created views: v_evaluation_summary, v_run_summary")

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

MLFLOW_EXPERIMENTS = {
    "yield":  "biox_yield_prediction",
    "uplift": "biox_uplift_estimation",
    "rec":    "biox_recommendation_eval",
}
for name in MLFLOW_EXPERIMENTS.values():
    mlflow.set_experiment(name)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiments ready:", list(MLFLOW_EXPERIMENTS.values()))

## 8. Tracking helper functions

These helpers write to both Postgres (`biox_experiments`) and MLflow, keeping the two stores in sync:

- `create_project`, `create_experiment` — metadata rows
- `run_experiment` — trains a model (parent MLflow run) → writes `experiment_runs`, `run_parameters`, `run_artifacts`
- `evaluate_run` — scores a trained model under a scenario (nested MLflow run) → writes `evaluations`, `evaluation_metrics`, `evaluation_artifacts`, and rule/subset rows

In [ ]:
import pickle

# in-memory registry of trained models so evaluate_run can reuse them without retraining
_TRAINED_MODELS = {}   # run_id -> {"pipe": ..., "meta": ..., "kind": "yield"|"uplift"}


def _scalar(conn, sql, **params):
    return conn.execute(text(sql), params).scalar()


def create_project(name, description="", crop=CROP, product=PRODUCT,
                   region=REGION, created_by="biox_ds"):
    with engine.begin() as conn:
        pid = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.projects (name, description, crop, product, region, created_by)
            VALUES (:n, :d, :c, :p, :r, :u) RETURNING project_id
        """, n=name, d=description, c=crop, p=product, r=region, u=created_by)
    return int(pid)


def create_experiment(project_id, name, hypothesis="", model_family="",
                      description="", created_by="biox_ds", status="running"):
    with engine.begin() as conn:
        eid = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.experiments
              (project_id, name, hypothesis, model_family, description, created_by, status)
            VALUES (:pid, :n, :h, :mf, :d, :u, :s) RETURNING experiment_id
        """, pid=project_id, n=name, h=hypothesis, mf=model_family,
             d=description, u=created_by, s=status)
    return int(eid)


def get_treatment_config(product="BioX-A", dose=2500, method="foliar",
                         stage="flowering", scenario="standard"):
    with engine.begin() as conn:
        cid = conn.execute(text(f"""
            SELECT config_id FROM {EXP_SCHEMA}.treatment_configs
            WHERE product=:p AND stress_scenario=:s LIMIT 1
        """), {"p": product, "s": scenario}).scalar()
        if cid is None:
            cid = _scalar(conn, f"""
                INSERT INTO {EXP_SCHEMA}.treatment_configs
                  (product, dose_ml_ha, application_method, crop_stage, stress_scenario, description)
                VALUES (:p, :dose, :m, :st, :s, :desc) RETURNING config_id
            """, p=product, dose=dose, m=method, st=stage, s=scenario,
                 desc=f"{product} {scenario} config")
    return int(cid)

print("Metadata helpers defined: create_project, create_experiment, get_treatment_config")

In [ ]:
import shap
import matplotlib
matplotlib.use("Agg")  # headless for artifact generation


def run_experiment(experiment_id, config):
    """Train ONE model (a Run = parent MLflow run). Returns run_id.

    config keys: kind ('yield'|'uplift'), model_type, feature_set, params, notes
    """
    kind = config.get("kind", "yield")
    model_type = config.get("model_type", "random_forest")
    feature_set = config.get("feature_set", "full")
    params = config.get("params", {})
    mlflow_exp = MLFLOW_EXPERIMENTS["yield" if kind == "yield" else "uplift"]
    mlflow.set_experiment(mlflow_exp)

    start = datetime.utcnow()
    with mlflow.start_run(run_name=f"{kind}_{model_type}_{feature_set}") as parent:
        mlflow.log_params({"kind": kind, "model_type": model_type,
                           "feature_set": feature_set, "random_seed": RANDOM_SEED,
                           **{f"hp_{k}": v for k, v in params.items()}})

        artifacts = []
        if kind == "yield":
            pipe, meta = train_yield_model(feature_store, model_type, feature_set, params)
            mlflow.sklearn.log_model(pipe, "model")
            # SHAP + feature importance artifacts (best-effort)
            try:
                Xs, _, _, _, _ = get_Xy(feature_store, feature_set, include_treatment=True)
                Xt = pipe.named_steps["prep"].transform(Xs.iloc[:200])
                if hasattr(Xt, "toarray"):
                    Xt = Xt.toarray()
                expl = shap.Explainer(pipe.named_steps["model"], Xt)
                sv = expl(Xt)
                fig = plt.figure()
                shap.summary_plot(sv, Xt, show=False)
                p = os.path.join(ARTIFACT_DIR, f"shap_{parent.info.run_id}.png")
                plt.tight_layout(); plt.savefig(p, dpi=90, bbox_inches="tight"); plt.close()
                mlflow.log_artifact(p); artifacts.append(("shap_summary", p, "figure"))
            except Exception as ex:
                print("  (SHAP skipped:", ex, ")")
            _TRAINED_MODELS_kind = "yield"
        else:
            upl = estimate_uplift(feature_store, method=model_type, feature_set=feature_set, params=params)
            pipe, meta = upl, {"model_type": model_type, "feature_set": feature_set,
                               "params": params, "method": upl["method"]}
            # persist the uplift model object as a pickle artifact
            p = os.path.join(ARTIFACT_DIR, f"uplift_{parent.info.run_id}.pkl")
            with open(p, "wb") as f:
                pickle.dump({"segment_estimates": upl["segment_estimates"], "ate": upl["ate"]}, f)
            mlflow.log_artifact(p); artifacts.append(("uplift_model", p, "model"))
            _TRAINED_MODELS_kind = "uplift"

        run_mlflow_id = parent.info.run_id
        end = datetime.utcnow()

    # mirror into Postgres
    with engine.begin() as conn:
        run_id = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.experiment_runs
              (experiment_id, mlflow_run_id, mlflow_experiment_name, model_type,
               start_time, end_time, status, notes)
            VALUES (:eid, :mru, :men, :mt, :st, :et, 'completed', :notes)
            RETURNING run_id
        """, eid=experiment_id, mru=run_mlflow_id, men=mlflow_exp, mt=model_type,
             st=start, et=end, notes=config.get("notes", ""))
        # params
        for k, v in {"model_type": model_type, "feature_set": feature_set,
                     "kind": kind, **{f"hp_{k}": v for k, v in params.items()}}.items():
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.run_parameters (run_id, param_name, param_value, param_type)
                VALUES (:r, :n, :v, :t)
            """), {"r": run_id, "n": k, "v": str(v), "t": type(v).__name__})
        # artifacts
        for name, path, atype in artifacts:
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.run_artifacts (run_id, artifact_name, artifact_path, artifact_type)
                VALUES (:r, :n, :p, :t)
            """), {"r": run_id, "n": name, "p": path, "t": atype})

    _TRAINED_MODELS[run_id] = {"pipe": pipe, "meta": meta, "kind": _TRAINED_MODELS_kind}
    print(f"  [run {run_id}] trained {kind}/{model_type}/{feature_set}  (mlflow={run_mlflow_id[:8]})")
    return run_id

print("run_experiment defined.")

In [ ]:
def _subset_for_scenario(scenario):
    """Return (dataframe, cv_strategy, filter_description) for a named scenario."""
    df = feature_store
    if scenario == "leave_site_out":
        return df, "leave_site_out", ("all", None, None)
    if scenario == "holdout_site_S3":
        return df, "random_80_20", ("holdout", None, None)   # site handling inside eval
    if scenario == "moderate_stress_only":
        return df[df["stress_regime"] == "moderate_stress"], "random_80_20", ("moderate", "moderate_stress", None)
    if scenario.startswith("recommendation"):
        return df, None, ("all", None, None)
    return df, "leave_site_out", ("all", None, None)


def evaluate_run(run_id, scenario, rules=None):
    """Score a trained model (Evaluation = nested MLflow run). Returns evaluation_id.

    Metrics attach to the evaluation. Uplift evals log uplift_bias vs ground truth;
    recommendation evals log precision/recall/roi_accuracy.
    """
    if run_id not in _TRAINED_MODELS:
        raise ValueError(f"run {run_id} not in memory; re-run run_experiment first")
    entry = _TRAINED_MODELS[run_id]
    kind = entry["kind"]
    parent_mlflow = pd.read_sql(
        f"SELECT mlflow_run_id, mlflow_experiment_name FROM {EXP_SCHEMA}.experiment_runs WHERE run_id={run_id}",
        engine).iloc[0]
    mlflow.set_experiment(parent_mlflow["mlflow_experiment_name"])

    dsub, cv_strategy, (stress_filter, stress_lvl, stage_flt) = _subset_for_scenario(scenario)
    metrics, fold_df, artifacts = {}, None, []

    with mlflow.start_run(run_id=parent_mlflow["mlflow_run_id"]):
        with mlflow.start_run(run_name=f"eval::{scenario}", nested=True) as child:
            mlflow.log_params({"eval_scenario": scenario, "cv_strategy": str(cv_strategy)})

            if kind == "yield":
                res = evaluate_yield(dsub, entry["meta"]["model_type"],
                                     entry["meta"]["feature_set"],
                                     entry["meta"].get("params"),
                                     cv_strategy or "leave_site_out")
                metrics.update(res["aggregate"])
                fold_df = res["fold_metrics"]
                # predicted vs actual artifact
                fig = plt.figure(figsize=(5, 5))
                plt.scatter(res["truth"], res["predictions"], alpha=0.4, s=12)
                lims = [min(res["truth"]), max(res["truth"])]
                plt.plot(lims, lims, "k--"); plt.xlabel("actual"); plt.ylabel("predicted")
                plt.title(f"Yield pred vs actual [{scenario}]")
                p = os.path.join(ARTIFACT_DIR, f"pva_{child.info.run_id}.png")
                plt.tight_layout(); plt.savefig(p, dpi=90); plt.close()
                artifacts.append(("pred_vs_actual", p, "figure"))

            elif kind == "uplift":
                upl = estimate_uplift(dsub, method=entry["meta"]["model_type"],
                                      feature_set=entry["meta"]["feature_set"])
                m, seg_df = uplift_metrics(upl, dsub)
                metrics.update({
                    "ate_estimate": m["ate_estimate"],
                    "treatment_effect_error_mae": m["treatment_effect_error_mae"],
                    "est_uplift_moderate": m["est_uplift_moderate"],
                    "uplift_bias": m["est_uplift_moderate"] - m["true_uplift_moderate"],
                })
                # recommendation sub-metrics if this is a recommendation scenario
                if scenario.startswith("recommendation"):
                    rec_df, used_rules = recommend_fields(dsub, upl, rules)
                    rm = recommendation_metrics(rec_df)
                    metrics.update(rm)
                    # roi_accuracy: fraction of recommended fields that truly benefit
                    rec_true = rec_df[rec_df["recommend"]]
                    metrics["roi_accuracy"] = float((rec_true["true_effect"] > 0.10).mean()) if len(rec_true) else 0.0
                # true-vs-estimated segment artifact
                fig = plt.figure(figsize=(5, 4))
                plt.bar(seg_df["segment"], seg_df["true_uplift"], alpha=0.5, label="true")
                plt.bar(seg_df["segment"], seg_df["est_uplift"], alpha=0.5, label="estimated")
                plt.xticks(rotation=25); plt.legend(); plt.title(f"Uplift by segment [{scenario}]")
                p = os.path.join(ARTIFACT_DIR, f"uplift_seg_{child.info.run_id}.png")
                plt.tight_layout(); plt.savefig(p, dpi=90); plt.close()
                artifacts.append(("uplift_by_segment", p, "figure"))

            # log metrics to MLflow (child)
            for k, v in metrics.items():
                if v is not None and not (isinstance(v, float) and np.isnan(v)):
                    mlflow.log_metric(k, float(v))
            child_mlflow_id = child.info.run_id
            for _, path, _ in [(a[0], a[1], a[2]) for a in artifacts]:
                mlflow.log_artifact(path)

    # mirror into Postgres
    cfg_id = get_treatment_config(scenario=scenario if scenario.startswith("recommendation") else "standard")
    with engine.begin() as conn:
        eval_id = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.evaluations
              (run_id, mlflow_run_id, eval_scenario, cv_strategy, treatment_config_id, status)
            VALUES (:r, :m, :s, :cv, :cfg, 'completed') RETURNING evaluation_id
        """, r=run_id, m=child_mlflow_id, s=scenario, cv=str(cv_strategy), cfg=cfg_id)

        # aggregate metrics (fold IS NULL)
        for k, v in metrics.items():
            if v is None or (isinstance(v, float) and np.isnan(v)):
                continue
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.evaluation_metrics (evaluation_id, metric_name, metric_value, fold)
                VALUES (:e, :n, :v, NULL)
            """), {"e": eval_id, "n": k, "v": float(v)})
        # per-fold metrics for yield
        if fold_df is not None:
            for _, fr in fold_df.iterrows():
                for mname in ["rmse", "mae", "r2"]:
                    conn.execute(text(f"""
                        INSERT INTO {EXP_SCHEMA}.evaluation_metrics (evaluation_id, metric_name, metric_value, fold)
                        VALUES (:e, :n, :v, :f)
                    """), {"e": eval_id, "n": mname, "v": float(fr[mname]), "f": int(fr["fold"])})
        # eval artifacts
        for name, path, atype in artifacts:
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.evaluation_artifacts (evaluation_id, artifact_name, artifact_path, artifact_type)
                VALUES (:e, :n, :p, :t)
            """), {"e": eval_id, "n": name, "p": path, "t": atype})
        # recommendation rules + field subset
        if rules is not None or scenario.startswith("recommendation"):
            used = {**DEFAULT_RULES, **(rules or {})}
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.recommendation_rules
                  (evaluation_id, uplift_threshold, roi_threshold, confidence_lower_bound,
                   allowed_crop_stages, max_soil_ec, description)
                VALUES (:e, :ut, :rt, :cb, :st, :ec, :d)
            """), {"e": eval_id, "ut": used["uplift_threshold"], "rt": used["roi_threshold"],
                   "cb": used["confidence_lower_bound"],
                   "st": json.dumps(used["allowed_crop_stages"]),
                   "ec": used["max_soil_ec"], "d": f"rules for {scenario}"})
        conn.execute(text(f"""
            INSERT INTO {EXP_SCHEMA}.field_subsets
              (evaluation_id, site_ids, stress_level_filter, crop_stage_filter, n_fields)
            VALUES (:e, :sids, :slf, :csf, :n)
        """), {"e": eval_id, "sids": json.dumps(sorted(dsub["site_id"].unique().tolist())),
               "slf": stress_lvl, "csf": stage_flt, "n": int(len(dsub))})

    print(f"    [eval {eval_id}] run={run_id} scenario={scenario} "
          + " ".join(f"{k}={v:.3f}" for k, v in metrics.items() if isinstance(v, (int, float)))[:120])
    return eval_id

print("evaluate_run defined.")

## 9. Run genuine experiments

Create a project and experiments, train real models, and evaluate each under multiple scenarios. These produce **genuine** MLflow parent/child runs and Postgres rows.

In [ ]:
# --- Project ---
project_id = create_project(
    "BioX-A Tomato Tropical Asia",
    description="POC: identify fields where BioX-A improves tomato yield under moderate water stress.",
)
print("Project:", project_id)

# --- Experiment 1: yield prediction model comparison ---
exp_yield = create_experiment(project_id, "Yield model comparison",
                              hypothesis="Tree models beat linear baseline on leave-site-out yield RMSE.",
                              model_family="yield_prediction")

yield_run_ids = {}
for mt in ["linear_regression", "random_forest", "xgboost"]:
    rid = run_experiment(exp_yield, {"kind": "yield", "model_type": mt, "feature_set": "full"})
    yield_run_ids[mt] = rid
    for scenario in ["leave_site_out", "moderate_stress_only"]:
        evaluate_run(rid, scenario)

# --- Experiment 2: uplift estimation (OLS vs causal forest) ---
exp_uplift = create_experiment(project_id, "Uplift: OLS causal vs Causal Forest",
                               hypothesis="Both recover +0.45 t/ha moderate-stress uplift; CF has lower CATE error.",
                               model_family="uplift_estimation")

uplift_run_ids = {}
for mt in ["ols_causal", "causal_forest"]:
    rid = run_experiment(exp_uplift, {"kind": "uplift", "model_type": mt, "feature_set": "full"})
    uplift_run_ids[mt] = rid
    for scenario in ["leave_site_out", "moderate_stress_only"]:
        evaluate_run(rid, scenario)

# --- Experiment 3: recommendation ROI-threshold sensitivity ---
exp_rec = create_experiment(project_id, "Recommendation ROI-threshold sensitivity",
                            hypothesis="Higher ROI thresholds increase precision, reduce recall.",
                            model_family="recommendation_eval")
rec_run = run_experiment(exp_rec, {"kind": "uplift", "model_type": "ols_causal", "feature_set": "full"})
for roi_thr in [1.5, 2.0, 2.5]:
    evaluate_run(rec_run, f"recommendation_roi_{roi_thr}", rules={"roi_threshold": roi_thr})

print("\nGenuine runs complete.")

In [ ]:
# --- Register best models to the MLflow Model Registry (Staging -> Production) ---
from mlflow.tracking import MlflowClient
client = MlflowClient()

# best yield run = lowest leave-site-out RMSE among yield runs
best_yield = pd.read_sql(f"""
    SELECT run_id, model_type, rmse FROM {EXP_SCHEMA}.v_evaluation_summary
    WHERE eval_scenario='leave_site_out' AND rmse IS NOT NULL
      AND model_type IN ('linear_regression','random_forest','xgboost')
    ORDER BY rmse ASC LIMIT 1
""", engine)

def _register(run_id_pg, registry_name):
    mru = pd.read_sql(
        f"SELECT mlflow_run_id FROM {EXP_SCHEMA}.experiment_runs WHERE run_id={run_id_pg}",
        engine)["mlflow_run_id"].iloc[0]
    try:
        mv = mlflow.register_model(f"runs:/{mru}/model", registry_name)
        client.transition_model_version_stage(registry_name, mv.version, "Production",
                                               archive_existing_versions=True)
        print(f"Registered {registry_name} v{mv.version} (Production) from pg run {run_id_pg}")
    except Exception as ex:
        print(f"  (registry step for {registry_name} skipped: {ex})")

if len(best_yield):
    print("Best yield run:", best_yield.iloc[0].to_dict())
    _register(int(best_yield.iloc[0]["run_id"]), "biox_yield_model")

# best uplift run = smallest |uplift_bias|
best_uplift = pd.read_sql(f"""
    SELECT run_id, model_type, uplift_bias FROM {EXP_SCHEMA}.v_evaluation_summary
    WHERE uplift_bias IS NOT NULL AND model_type IN ('ols_causal','causal_forest')
    ORDER BY ABS(uplift_bias) ASC LIMIT 1
""", engine)
if len(best_uplift):
    print("Best uplift run:", best_uplift.iloc[0].to_dict())
    # uplift models were pickled as artifacts, not logged as sklearn flavor -> note only
    print("  (uplift model tracked via artifact pickle; registry entry optional in POC)")

## 10. Supplemental mock history

Adds extra experiments/runs/evaluations (clearly flagged as `mock`) so the dashboard shows a fuller history. These are metadata-only rows with plausible, internally-consistent metrics — they do not train real models.

In [ ]:
mrng = np.random.default_rng(7)

# a second project so the multi-project view is populated
project2 = create_project("BioX-A Rice Salinity Study",
                          description="Exploratory: BioX-A response on rice under salinity (mock history).",
                          crop="rice")

mock_experiments = [
    (project_id, "Random Forest hyperparameter sweep", "random_forest tuning", "yield_prediction"),
    (project_id, "XGBoost vs LightGBM yield comparison", "boosting comparison", "yield_prediction"),
    (project_id, "Feature-set ablation (satellite vs none)", "satellite value", "yield_prediction"),
    (project2,   "Stress-segment CATE analysis (rice)", "salinity CATE", "uplift_estimation"),
    (project2,   "Double ML pilot (rice)", "double ML", "uplift_estimation"),
]

def _insert_mock_run(experiment_id, model_type, params):
    with engine.begin() as conn:
        rid = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.experiment_runs
              (experiment_id, mlflow_run_id, mlflow_experiment_name, model_type,
               start_time, end_time, status, notes)
            VALUES (:e, :m, :men, :mt, now(), now(), 'completed', 'mock')
            RETURNING run_id
        """, e=experiment_id, m=f"mock-{mrng.integers(1e6,9e6)}",
             men="mock", mt=model_type)
        for k, v in params.items():
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.run_parameters (run_id, param_name, param_value, param_type)
                VALUES (:r,:n,:v,:t)
            """), {"r": rid, "n": k, "v": str(v), "t": type(v).__name__})
    return rid

def _insert_mock_eval(run_id, scenario, metrics):
    with engine.begin() as conn:
        eid = _scalar(conn, f"""
            INSERT INTO {EXP_SCHEMA}.evaluations (run_id, mlflow_run_id, eval_scenario, cv_strategy, status)
            VALUES (:r, :m, :s, 'leave_site_out', 'completed') RETURNING evaluation_id
        """, r=run_id, m=f"mock-{mrng.integers(1e6,9e6)}", s=scenario)
        for k, v in metrics.items():
            conn.execute(text(f"""
                INSERT INTO {EXP_SCHEMA}.evaluation_metrics (evaluation_id, metric_name, metric_value, fold)
                VALUES (:e,:n,:v,NULL)
            """), {"e": eid, "n": k, "v": float(v)})
    return eid

n_runs = n_evals = 0
for (pid, ename, hyp, fam) in mock_experiments:
    eid = create_experiment(pid, ename, hypothesis=hyp, model_family=fam, status="completed")
    for _ in range(mrng.integers(3, 6)):  # 3-5 runs per experiment
        if fam == "yield_prediction":
            mt = mrng.choice(["random_forest", "xgboost", "lightgbm"])
            rid = _insert_mock_run(eid, mt, {"n_estimators": int(mrng.choice([200,300,400])),
                                             "max_depth": int(mrng.choice([4,6,8])),
                                             "feature_set": mrng.choice(["with_satellite","full"])})
            n_runs += 1
            for scen in ["leave_site_out", "moderate_stress_only"]:
                r2 = float(np.clip(mrng.normal(0.82, 0.05), 0.68, 0.93))
                rmse = float(np.clip(mrng.normal(0.55, 0.08), 0.35, 0.9))
                _insert_mock_eval(rid, scen, {"r2": r2, "rmse": rmse, "mae": rmse*0.8})
                n_evals += 1
        else:
            mt = mrng.choice(["ols_causal", "causal_forest", "double_ml"])
            rid = _insert_mock_run(eid, mt, {"feature_set": "full"})
            n_runs += 1
            for scen in ["leave_site_out", "moderate_stress_only"]:
                bias = float(mrng.normal(0.0, 0.05))
                _insert_mock_eval(rid, scen, {
                    "est_uplift_moderate": TRUE_EFFECT_MODERATE + bias,
                    "uplift_bias": bias,
                    "treatment_effect_error_mae": abs(bias) + float(mrng.uniform(0.02, 0.06)),
                    "recommendation_precision": float(np.clip(mrng.normal(0.8, 0.07), 0.5, 0.97)),
                    "recommendation_recall": float(np.clip(mrng.normal(0.72, 0.08), 0.4, 0.95)),
                })
                n_evals += 1

# a couple of saved comparisons referencing real evaluations
real_evals = pd.read_sql(f"SELECT evaluation_id FROM {EXP_SCHEMA}.evaluations ORDER BY evaluation_id LIMIT 4", engine)
with engine.begin() as conn:
    conn.execute(text(f"""
        INSERT INTO {EXP_SCHEMA}.experiment_comparisons (name, scope, evaluation_ids, created_by, notes)
        VALUES (:n, 'evaluation', :ids, 'biox_ds', :notes)
    """), {"n": "Yield models @ leave-site-out", "ids": json.dumps(real_evals["evaluation_id"].tolist()),
           "notes": "Auto-seeded comparison"})

print(f"Mock history added: {n_runs} runs, {n_evals} evaluations across {len(mock_experiments)} experiments (2 projects).")

## 11. Compare, reproduce, and diff

Query the tracking layer to answer real questions: best evaluation per experiment, ground-truth recovery, reproduce a run, and a recommendation diff between two ROI thresholds.

In [ ]:
# Best yield evaluation per experiment (leave-site-out)
best_per_exp = pd.read_sql(f"""
    SELECT experiment_name, model_type, eval_scenario,
           ROUND(rmse::numeric,3) AS rmse, ROUND(r2::numeric,3) AS r2
    FROM {EXP_SCHEMA}.v_evaluation_summary
    WHERE rmse IS NOT NULL
    ORDER BY experiment_name, rmse ASC
""", engine)
print("Yield evaluations (sorted by RMSE):")
best_per_exp.head(15)

In [ ]:
# Ground-truth recovery: estimated vs true moderate-stress uplift (genuine uplift runs)
gt = pd.read_sql(f"""
    SELECT experiment_name, model_type, eval_scenario,
           ROUND(est_uplift_moderate::numeric,3) AS est_uplift_moderate,
           ROUND(uplift_bias::numeric,3) AS uplift_bias,
           ROUND(treatment_effect_error::numeric,3) AS te_error
    FROM {EXP_SCHEMA}.v_evaluation_summary
    WHERE est_uplift_moderate IS NOT NULL
    ORDER BY ABS(uplift_bias) ASC
""", engine)
print(f"KNOWN synthetic moderate-stress uplift = +{TRUE_EFFECT_MODERATE:.2f} t/ha\n")
print("Model recovery (sorted by |bias|):")
gt

In [ ]:
# Reproduce a run: pull its params from Postgres, re-train + re-evaluate, confirm metrics match
def reproduce_run(run_id):
    params = pd.read_sql(f"""
        SELECT param_name, param_value FROM {EXP_SCHEMA}.run_parameters WHERE run_id={run_id}
    """, engine).set_index("param_name")["param_value"].to_dict()
    kind = params.get("kind", "yield")
    model_type = params.get("model_type", "random_forest")
    feature_set = params.get("feature_set", "full")
    if kind == "yield":
        res = evaluate_yield(feature_store, model_type, feature_set, None, "leave_site_out")
        return {"kind": kind, "model_type": model_type, **{k: round(v,4) for k,v in res["aggregate"].items()}}
    else:
        upl = estimate_uplift(feature_store, model_type, feature_set)
        m, _ = uplift_metrics(upl, feature_store)
        return {"kind": kind, "model_type": model_type,
                "est_uplift_moderate": round(m["est_uplift_moderate"],4)}

rf_run = yield_run_ids["random_forest"]
print(f"Reproducing run {rf_run} (should match its original leave-site-out metrics):")
print(reproduce_run(rf_run))

In [ ]:
# Recommendation diff: same trained uplift model, two ROI thresholds -> which fields flip?
upl_full = estimate_uplift(feature_store, "ols_causal", "full")
rec_low, _ = recommend_fields(feature_store, upl_full, rules={"roi_threshold": 1.5})
rec_high, _ = recommend_fields(feature_store, upl_full, rules={"roi_threshold": 2.5})

diff = rec_low[["field_id", "stress_regime", "roi", "recommend"]].merge(
    rec_high[["field_id", "recommend"]], on="field_id", suffixes=("_roi15", "_roi25"))
flipped = diff[diff["recommend_roi15"] != diff["recommend_roi25"]]

print(f"ROI 1.5x recommends: {int(rec_low['recommend'].sum())} fields")
print(f"ROI 2.5x recommends: {int(rec_high['recommend'].sum())} fields")
print(f"Fields that flip (apply -> do-not-apply when threshold rises): {len(flipped)}")
flipped.head(10)

## 12. Final field-level recommendations + AI agronomist summary

Produce the deliverable output: per-field predicted yield, uplift %, ROI, confidence, apply/do-not-apply, key drivers, and satellite evidence. Persist to a `field_recommendations` table for the dashboard. All language is framed as synthetic-POC evidence.

In [ ]:
# Build final recommendations using the best uplift model + default rules
final_rec, final_rules = recommend_fields(feature_store, upl_full, rules=DEFAULT_RULES)

# join a predicted yield from the best yield model
best_yield_run = int(best_yield.iloc[0]["run_id"]) if len(best_yield) else yield_run_ids["random_forest"]
best_pipe = _TRAINED_MODELS[best_yield_run]["pipe"]
Xall, _, _, _, _ = get_Xy(feature_store, _TRAINED_MODELS[best_yield_run]["meta"]["feature_set"], True)
final_rec["predicted_yield_t_ha"] = best_pipe.predict(Xall)

# persist for the dashboard
final_rec.to_sql("field_recommendations", engine, if_exists="replace", index=False)
print(f"Wrote field_recommendations ({len(final_rec)} rows). "
      f"Recommended: {int(final_rec['recommend'].sum())}")

# AI agronomist narrative generator (template-based, deterministic, hedged)
def agronomist_summary(row):
    verdict = "recommended for BioX-A" if row["recommend"] else "not recommended for BioX-A"
    conf = row["confidence"]
    signals = []
    if row["stress_regime"] == "moderate_stress": signals.append("moderate water stress")
    if "positive NDWI" in row["key_drivers"]: signals.append("positive post-treatment NDWI recovery")
    if "positive NDRE" in row["key_drivers"]: signals.append("positive post-treatment NDRE recovery")
    if "below-median" in row["key_drivers"]: signals.append("below-median pre-treatment NDRE")
    sig = ", ".join(signals) if signals else "model-estimated response profile"
    return (f"Based on synthetic POC evidence and model behavior, field {row['field_id']} is {verdict}. "
            f"Expected yield uplift is {row['uplift_pct']:.1f}% ({row['pred_uplift_t_ha']:+.2f} t/ha), "
            f"expected ROI is {row['roi']:.1f}x, confidence is {conf}. "
            f"Supporting signals include {sig}. "
            f"This reflects platform-recovered response patterns on synthetic data, not proven product efficacy.")

# show an example for a recommended moderate-stress field
ex = final_rec[(final_rec["recommend"]) & (final_rec["stress_regime"] == "moderate_stress")]
if len(ex):
    print("\n--- Example AI agronomist recommendation ---")
    print(agronomist_summary(ex.iloc[0]))

final_rec[final_rec["recommend"]].head(8)[
    ["field_id", "site_id", "stress_regime", "predicted_yield_t_ha",
     "pred_uplift_t_ha", "uplift_pct", "roi", "confidence"]]

In [ ]:
# Final layer counts -- confirms the experiments hierarchy is populated end-to-end
summary = {}
for t in ["projects", "experiments", "experiment_runs", "evaluations",
          "evaluation_metrics", "run_parameters", "evaluation_artifacts",
          "recommendation_rules", "field_subsets", "experiment_comparisons"]:
    summary[t] = pd.read_sql(f"SELECT COUNT(*) n FROM {EXP_SCHEMA}.{t}", engine)["n"].iloc[0]

print("biox_experiments layer row counts:")
for k, v in summary.items():
    print(f"  {k:24s}: {v}")

print("\nDone. Launch the dashboard with:")
print("  streamlit run biox_experiments_dashboard.py")